# EDGE Class: End-to-End ML Deployment Demo
This notebook creates a dummy dataset, trains a model, and exports a .pkl file for deployment.

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
rng = np.random.default_rng(42)
n_rows = 120

study_hours = rng.uniform(0.5, 8.0, size=n_rows)
attendance = rng.integers(45, 100, size=n_rows)
assignments_completed = rng.integers(0, 11, size=n_rows)

score = (
    0.5 * study_hours
    + 0.03 * attendance
    + 0.25 * assignments_completed
    + rng.normal(0, 0.7, size=n_rows)
)
pass_exam = (score >= 5.5).astype(int)

df = pd.DataFrame(
    {
        'study_hours': study_hours.round(2),
        'attendance': attendance,
        'assignments_completed': assignments_completed,
        'pass_exam': pass_exam,
    }
)

os.makedirs('../data', exist_ok=True)
df.to_csv('../data/dummy_students.csv', index=False)
print(f"Saved dataset: ../data/dummy_students.csv | rows={len(df)}")
df.head()

,study_hours,attendance,assignments_completed,pass_exam
0,6.30,65,10,1
1,3.79,90,10,1
2,6.94,45,5,1
3,5.73,94,0,1
4,1.21,74,6,0


In [ ]:
features = ['study_hours', 'attendance', 'assignments_completed']
target = 'pass_exam'

X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df[target],
    test_size=0.2,
    random_state=42,
    stratify=df[target],
)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42)),
])
model.fit(X_train, y_train)

preds = model.predict(X_test)
accuracy = accuracy_score(y_test, preds)
print(f'Model accuracy: {accuracy:.3f}')
print(classification_report(y_test, preds))

Accuracy: 0.833
              precision    recall  f1-score   support

           0       0.86      0.80      0.83        15
           1       0.81      0.87      0.84        15

    accuracy                           0.83        30
   macro avg       0.83      0.83      0.83        30
weighted avg       0.83      0.83      0.83        30



In [5]:
os.makedirs('../models', exist_ok=True)
with open('../models/student_pass_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Saved: ../models/student_pass_model.pkl')

Saved: ../models/student_pass_model.pkl


In [ ]:
sample = pd.DataFrame([{
    'study_hours': 4.5,
    'attendance': 85,
    'assignments_completed': 7,
}])

pred = int(model.predict(sample)[0])
proba = float(model.predict_proba(sample)[0][1])

print('Prediction:', 'Pass' if pred == 1 else 'Fail')
print('Pass probability:', round(proba, 3))

Prediction: Pass
Pass probability: 0.865


: 